# R parity matrix
The tables compare supported operations in Python and R and identify
comparisons with upstream `mboost` and `gamboostLSS`.
## Core models
```{list-table}
:header-rows: 1
* - Feature
  - Python
  - R binding
  - Matches R `mboost`
* - `glmboost`, `gamboost`
  - Yes
  - Yes
  - Yes (≤1e-6)
* - `cvrisk` (bootstrap/kfold/subsampling)
  - Yes
  - Yes; model weights are combined with fold weights
  - Yes
* - stability selection
  - Yes
  - Yes
  - Yes (`stabs` math)
* - AIC / logLik / summary / varimp
  - Yes
  - Yes
  - Yes where defined
* - `hatvalues`, `confint`, `plot_risk`, `plot_varimp`
  - Yes
  - Yes
  - Yes
* - `stabsel`
  - Any fitted model
  - Linear only: it fits through `glmboost`, so base-learner terms refuse
  - Yes (`stabs` math)
```
## Base learners
`bols`, `bbs`, `bmono`, `btree`, `brandom`, `bspatial`, `bmrf`, and `bns`
are available in R formulas. `bmrf` resolves its required `K=` or `bnd=`
matrix from the formula environment. `bkernel` has no R equivalent.
## Families

The tables below are generated from `mboostrs.family_capabilities()` at
book-build time. A family added to the registry appears here on the next
rebuild with no docs edit.



In [1]:
import mboostrs as mb
import polars as pl

caps = mb.family_capabilities(output="polars")
standard = caps.filter(pl.col("kind").is_in(["standard", "standard_survival", "standard_multiclass"]))
standard.select(["name", "kind", "r_package", "r_name", "known_deviation"])

name,kind,r_package,r_name,known_deviation
str,str,str,str,str
"""AUC""","""standard""","""mboost""","""AUC""",""""""
"""AdaExp""","""standard""","""mboost""","""AdaExp""",""""""
"""Binomial""","""standard""","""mboost""","""Binomial""",""""""
"""Cindex""","""standard_survival""","""mboost""","""Cindex""",""""""
"""CoxPH""","""standard_survival""","""mboost""","""CoxPH""",""""""
…,…,…,…,…
"""Hurdle""","""standard""","""mboost""","""Hurdle""",""""""
"""Poisson""","""standard""","""mboost""","""Poisson""",""""""
"""PropOdds""","""standard""","""mboost""","""PropOdds""",""""""


## Distributional families

`known_deviation` records differences from R for each family. See
[Implementation differences](../implementation/parity_and_bugs) for their
mechanisms and the corresponding tests.


In [2]:
lss = caps.filter(pl.col("kind") == "lss")
lss.select(["name", "r_package", "r_name", "known_deviation"])

name,r_package,r_name,known_deviation
str,str,str,str
"""BCCGLSS""","""gamboostLSS""","""BCCGLSS""",""""""
"""BCPELSS""","""gamboostLSS""","""BCPELSS""",""""""
"""BCTLSS""","""gamboostLSS""","""BCTLSS""",""""""
"""BetaLSS""","""gamboostLSS""","""BetaLSS""",""""""
"""BinomialLSS""","""gamboostLSS""","""BinomialLSS""",""""""
…,…,…,…
"""WeibullLSS""","""gamboostLSS""","""WeibullLSS""","""Auto offset deliberately fixes…"
"""ZINBLSS""","""gamboostLSS""","""ZINBLSS""",""""""
"""ZIPoLSS""","""gamboostLSS""","""ZIPoLSS""",""""""


Rows with `kind == "standard_missing"` identify upstream families or
helpers that mboostrs has not implemented:


In [3]:
caps.filter(pl.col("kind") == "standard_missing").select(["name", "r_package", "notes"])

name,r_package,notes
str,str,str
"""FP""","""mboost""","""Known upstream mboost family/h…"
"""Family""","""mboost""","""Known upstream mboost family/h…"
"""IPCweights""","""mboost""","""Not a boosting family (no fit/…"
"""RCG""","""mboost""","""Known upstream mboost family/h…"


### Distributional fitting mechanics

```{list-table}
:header-rows: 1
* - Feature
  - Python
  - R binding
  - Note
* - cyclic fitting
  - Yes
  - Yes
  - `Bols`, difference-penalized `Bbs`, ridge-penalized `Bns`, random-intercept `Brandom`, graph-smoothed `Bmrf`, tensor-product `Bspatial`, shape-constrained `Bmono`, caller-constrained `Bconstrained`, and tree-based `Btree` may be mixed within each parameter
* - non-cyclic fitting
  - mixed `Bols`/`Bbs`/`Bns`/`Brandom`/`Bmrf`/`Bspatial`/`Bmono`/`Bconstrained`/`Btree`
  - mixed `Bols`/`Bbs`/`Bns`/`Brandom`/`Bmrf`/`Bspatial`/`Bmono`/`Bconstrained`/`Btree`
  - `gamboostLSS(method="noncyclic")` selects one parameter-qualified learner per global step
* - `cvrisk` (per-param mstop grid)
  - Yes
  - Yes
  - mixed structured learners; smoothing parameters and factorizations are recomputed from each fold's in-bag weights
* - stability selection
  - Yes
  - Yes
  - `stabsel_glmboostLSS` is linear; `stabsel_gamboostLSS` tracks every supported mixed learner with parameter-qualified identities
```
Where noted in the tables above, `mboostrs` implements the method independently
and differs from R; see [](../implementation/parity_and_bugs).


## Multinomial
```{list-table}
:header-rows: 1
* - Feature
  - Python
  - R binding
* - `bols` (linear) multinomial
  - Yes
  - Yes (matches R `mboost` ~1e-13)
* - `bbs` (smooth) multinomial
  - Yes
  - Yes (matches R `mboost` ~1e-6)
* - predict response matrix, in-sample
  - Yes
  - Yes
* - predict newdata / truncated mstop
  - Yes
  - Yes
```
The R binding supports **linear (`bols`) and smooth (`bbs`) multinomial**
`gamboost`, matching R `mboost`'s `bols/bbs(x) %O% bols(class)`.
newdata/truncated-mstop prediction is available in both Python and R.
## Multinomial limitations
- Factor-covariate `bols` and `btree`/`bmono` inside the multinomial `%O%`
  construction (a separate problem).

## Mixed distributional learners

Mixed LSS paths accept `Bols`, `Bbs`, `Bns`, `Brandom`, `Bmrf`, `Bspatial`, `Bmono`, `Bconstrained`, `Btree`, `Bkernel`, and `Buser`;
  `Brandom`, `Bmrf`, `Bspatial`, `Bmono`, and `Btree` fits and link predictions are checked against R `gamboostLSS`.
  Replay, CV, stability selection, and serialization have Python and R contract tests;
  grouped learners additionally test unknown-level behavior. `Bconstrained` is
  an mboostrs caller-matrix extension validated by feasibility and replay invariants.
  `Buser` has upstream fit/link-prediction parity plus explicit-matrix replay
  and resampling contracts in both frontends.
  `Bkernel` is operationally tested as an mboostrs kernel-ridge extension, not
  counted as parity with the upstream pathway/GWAS construction.
`gamboostLSS` accepts numeric and factor `by=` terms on `bbs` and `bols`.
The recorded fitting and prediction comparisons use a tolerance of 1e-10
against upstream `gamboostLSS`. Python mixed-LSS varying-coefficient
learners support fitting and training-data replay, but not new-data
prediction.
## Additional capabilities
```{list-table}
:header-rows: 1
* - Feature
  - Python
  - R binding
  - Note
* - Prediction contributions (`which=`, `aggregate=`)
  - Yes
  - —
  - Per-learner decomposition; GLM + GAM
* - Extraction methods (`variable_names`, `model_frame`, etc.)
  - Yes
  - —
  - R equivalent: `extract(what=...)`, `variable.names`, `model.frame`
* - Structured `summary(as_dict=True)`
  - Yes
  - —
  - JSON-serializable
* - Weighted LSS fits (`glmboostLSS(weights=)`)
  - Yes
  - —
  - Observation weights for distributional models
* - LSS GAIC / logLik with count-based df
  - Yes
  - Yes
  - `gaic(k=2)` matches R `logLik.mboostrs_glmboostLSS` df
* - Model comparison (`gaic_table`, `compare_models`)
  - Yes
  - —
  - Cross-model GAIC/risk comparison
* - Family scanning (`scan_lss_families`)
  - Yes
  - —
  - Collects failures instead of crashing
* - mstop profiling (`profile_mstop`)
  - Yes
  - —
  - Risk/GAIC at mstop grid points
* - R diff harness (`compare_to_r`)
  - Yes
  - —
  - Structured Python vs R parity reports
* - Distribution functions (pdf/cdf/quantile/simulate)
  - Yes
  - Yes
  - 12/22 LSS families; 7 with full cdf+quantile
* - Diagnostics module (`quantile_residuals`, `pit`, `q_stats`, etc.)
  - Yes
  - —
  - Works for GLM, GAM, and LSS
```


## Numerical comparisons

The capability manifest names the tests for each family and frontend.
The table below lists those comparisons. A `verified` record identifies
a reference value or implementation and a tolerance; `unverified` means
that no such comparison is recorded.


In [4]:
fit_evidence = caps.filter(pl.col("fit") == "yes").select(
    ["name", "kind", "python_fit_status", "r_fit_status", "validation_ids"]
).sort(["r_fit_status", "kind", "name"])
fit_evidence

name,kind,python_fit_status,r_fit_status,validation_ids
str,str,str,str,str
"""LogLogLSS""","""lss""","""intentional-deviation""","""intentional-deviation""","""deviation.loglog_offset; py.fa…"
"""WeibullLSS""","""lss""","""intentional-deviation""","""intentional-deviation""","""deviation.weibull_offset; py.f…"
"""Gehan""","""standard_survival""","""intentional-deviation""","""intentional-deviation""","""deviation.gehan_offset; parity…"
"""GaussianKnownVarLSS""","""lss""","""smoke""","""smoke""","""parity.r.gaussianknownvarlss; …"
"""BCCGLSS""","""lss""","""unverified""","""unverified""","""py.family.contract; r.family.c…"
…,…,…,…,…
"""ZINBLSS""","""lss""","""unverified""","""verified""","""parity.r.zinblss; py.family.co…"
"""ZIPoLSS""","""lss""","""verified""","""verified""","""parity.python.zipolss; parity.…"
"""Gaussian""","""standard""","""verified""","""verified""","""parity.python.gaussian; parity…"


The [R parity evidence workflow](https://github.com/fcela/mboostrs/actions/workflows/parity.yml) runs all tests carrying the `r_parity` marker every Monday and on manual request. Pull requests touching families, learners, CV, LSS, prediction, or bindings run the checked-in focused node list. Each run uploads `parity-results.json`, conforming to [`parity-result.schema.json`](../../schemas/parity-result.schema.json), with node status, measured absolute and relative error, the tolerance used by the numeric comparison, package versions, platform, and commit.

A test that does not report its numeric comparison has
`measurement_status = "not_emitted"` and null error/tolerance values. Read
those nulls as missing measurements, not zero error.
